# Eskadra Bielik — Misja 2 — wariant Google Colab

**RAG w oparciu o model Bielik i lokalną bazę wektorową — pełny warsztat na Colab z GPU.**

Ten notebook jest *kompletnym ekwiwalentem* 8-krokowego warsztatu z `README.md`, przeniesionym na Colab. Zachowuje strukturę, checkpointy i certyfikat — z tą różnicą, że zamiast Cloud Run + BigQuery używamy:

| Element warsztatu | Wariant GCP (oryginalny) | Wariant Colab (ten notebook) |
|---|---|---|
| Silnik LLM/embedding | Cloud Run + Ollama (GPU L4) | Ollama lokalnie + GPU T4 |
| Model LLM | SpeakLeash/bielik-4.5b-v3.0-instruct:Q8_0 | **bez zmian** |
| Model embedding | embeddinggemma | **bez zmian** |
| Vector store | BigQuery + VECTOR_SEARCH | ChromaDB lokalnie |
| Orchestrator | Cloud Run + FastAPI | FastAPI w wątku notebooka |
| Web UI | URL z Cloud Run | iframe z `serve_kernel_port_as_iframe` |
| Autoryzacja JWT | Google ID Token | brak (Ollama lokalnie) |
| Checkpointy/certyfikat | format `.enc` AES-256-CBC | **bez zmian** (ten sam openssl) |

## Co dostajesz

- Pełny system RAG po polsku z modelem Bielik 4.5B
- 8 checkpointów + certyfikat ukończenia (75 pkt) zgodne z wariantem GCP
- Web UI porównujący odpowiedź z RAG vs. bez RAG — w komórce notebooka

## Czego nie ma w wariancie Colab

- **Dashboard prowadzącego (Pub/Sub)** — wymaga GCP. Postępy widzisz lokalnie w punktach.
- **Zadania `🤖 Gemini CLI`** — odpadają, bo Gemini CLI to narzędzie Cloud Shell. Zastąpione komórkami Markdown z pytaniami otwartymi.
- **Skrypty `gcloud`** — nie są potrzebne. Wszystko lokalnie.


## Agenda warsztatu

| # | Temat | Czas | Punkty |
|---|---|---|:---:|
| 0 | Setup Colab i weryfikacja GPU | 3 min | **5** |
| 1 | Konfiguracja środowiska | 1 min | **10** |
| 2 | Ollama + Bielik + EmbeddingGemma | 10 min | **20** |
| 3 | Inicjalizacja lokalnej bazy wektorowej (ChromaDB) | 1 min | **5** |
| 4 | Orchestrator FastAPI (lokalny) | 2 min | **10** |
| 5 | Zasilanie bazy i pierwsze zapytania RAG | 5 min | **10** |
| 6 | Przegląd API — Swagger /docs | 3 min | **5** |
| 7 | Web UI w komórce notebooka | 5 min | **10** |
| 8 | Checkpointy + certyfikat ukończenia | 5 min | — |
| 9 | Sprzątanie | 1 min | — |
| | **Łącznie** | **~36 min** | **75 pkt** |

> 📌 **Punkty są przyznawane przez 8 checkpointów w sekcji 8.** Każdy checkpoint waliduje konkretną sekcję (CP1 → sekcja 0, CP2 → sekcja 1, CP3 → sekcja 2 itd.) i tworzy zaszyfrowany artefakt `cert_artifacts/checkpoint_N.enc` zgodny z wariantem GCP warsztatu.

> ⏱️ **Czasy są szacunkowe i zakładają pierwsze uruchomienie**: pobieranie modelu Bielik ~3–5 min, EmbeddingGemma poniżej minuty, pojedyncze zapytanie do Bielika 5–15 s na T4. Przy kolejnych uruchomieniach modele są cache'owane na dysku Colab.

> ⚠️ **WAŻNE — zanim ruszysz dalej:**
>
> 1. **Włącz GPU**: `Środowisko wykonawcze → Zmień typ środowiska → GPU (T4)`. Bez GPU model Bielik będzie odpowiadał kilka minut na zapytanie.
> 2. **Sesja Colab kończy się po ~12 h aktywnej pracy lub ~90 min bezczynności**. Modele Ollama (~5 GB) zostaną pobrane do `/root/.ollama/` i znikną po restarcie kernela — przejdź notebook za jednym posiedzeniem.
> 3. **Zaplanuj ~45 min**: pierwsze pobranie modelu Bielik trwa 3–5 min, EmbeddingGemma poniżej minuty, każde zapytanie do Bielika 5–15 s.
>
> Jeśli zobaczysz, że `nvidia-smi` w kroku 0 zwraca błąd — wróć do menu i włącz GPU.

## 0. Setup Colab i weryfikacja GPU `~3 min`

Odpowiednik kroku 1 warsztatu (Przygotowanie projektu Google Cloud) — w wariancie Colab tym co weryfikujemy jest GPU runtime i klon repo z hotel_rules.csv.

In [ ]:
# [0.1] Czy GPU jest aktywny?
!nvidia-smi

Spodziewany wynik: tabela z `Tesla T4` (lub `L4`/`A100` jeśli Colab Pro) i informacją o pamięci ~15 GB. Jeśli widzisz `command not found` lub `No devices` — wróć do menu i włącz GPU.

In [ ]:
# [0.2] Zależności systemowe Ollamy (wykrycie GPU + rozpakowanie instalatora)
!sudo apt-get install -y -qq pciutils lshw zstd

In [ ]:
# [0.3] Klon repo warsztatowego — potrzebujemy hotel_rules.csv, static/index.html, checkpointów
import os, pathlib

REPO_DIR = '/content/eskadra-bielik-misja2'
if not pathlib.Path(REPO_DIR).exists():
    !git clone --depth 1 https://github.com/Legard777/eskadra-bielik-misja2.git {REPO_DIR}
else:
    print(f'Repo już sklonowane w {REPO_DIR}')

os.chdir(REPO_DIR)
!ls -la

## 1. Konfiguracja środowiska `~1 min`

Odpowiednik kroku 2 (`source setup_env.sh`). Zmienne pythonowe zamiast bash exportów.

In [ ]:
# [1.1] Konfiguracja warsztatu
# WORKSHOP_EMAIL i WORKSHOP_PROJECT są używane jako materiał do klucza szyfrującego
# checkpointy — taki sam mechanizm jak w wariancie GCP (gcloud account + project).

WORKSHOP_EMAIL = 'uczestnik@example.com'   # ⬅️ ZMIEŃ na swój email
WORKSHOP_PROJECT = 'colab-warsztat-bielik' # placeholder zamiast project_id z GCP

MODEL_LLM = 'SpeakLeash/bielik-4.5b-v3.0-instruct:Q8_0'
MODEL_EMB = 'embeddinggemma'
OLLAMA_URL = 'http://localhost:11434'
ORCH_PORT = 8080
CHROMA_DIR = '/content/chroma_db'
COLLECTION_NAME = 'hotel_rules'
STATIC_DIR = f'{REPO_DIR}/orchestration/static'
CSV_PATH = f'{REPO_DIR}/vector_store/hotel_rules.csv'
CERT_DIR = f'{REPO_DIR}/cert_artifacts'

import os
os.makedirs(CHROMA_DIR, exist_ok=True)
os.makedirs(CERT_DIR, exist_ok=True)

print('Konfiguracja gotowa:')
for k, v in [('WORKSHOP_EMAIL', WORKSHOP_EMAIL), ('WORKSHOP_PROJECT', WORKSHOP_PROJECT),
             ('MODEL_LLM', MODEL_LLM), ('MODEL_EMB', MODEL_EMB),
             ('CHROMA_DIR', CHROMA_DIR), ('CERT_DIR', CERT_DIR)]:
    print(f'  {k:20s} = {v}')

> 📌 **Email decyduje o kluczu szyfrowania certyfikatu** — jeśli zmienisz email po wygenerowaniu części checkpointów, kolejne checkpointy będą szyfrowane innym kluczem i certyfikat nie zostanie odszyfrowany jednym hasłem. Ustaw email **raz, na samym początku**.

## 2. Ollama + Bielik + EmbeddingGemma `~10 min`

Odpowiednik kroku 3 warsztatu (Cloud Run + Ollama z modelami). Tu instalujemy Ollamę lokalnie w Colab i pobieramy oba modele bezpośrednio z Ollama Hub.

In [ ]:
# [2.1] Instalacja Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# [2.2] Uruchomienie Ollama serve w tle
import subprocess, time, requests

# Sprawdź czy nie działa już
try:
    requests.get(f'{OLLAMA_URL}/api/tags', timeout=2)
    print('Ollama już działa.')
except Exception:
    ollama_proc = subprocess.Popen(
        ['ollama', 'serve'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    # Czekamy aż złapie port 11434
    for _ in range(30):
        try:
            requests.get(f'{OLLAMA_URL}/api/tags', timeout=1)
            print('Ollama wystartowała.')
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError('Ollama nie wystartowała w 30 s — sprawdź logi')

# Weryfikacja: czy widzi GPU?
import json
tags = requests.get(f'{OLLAMA_URL}/api/tags').json()
print('Modele już dostępne:', [m['name'] for m in tags.get('models', [])] or '(jeszcze żadne)')

In [ ]:
# [2.3] Pobranie modelu Bielik 4.5B Q8_0 (~4.7 GB) — pierwsze pobranie 3–5 min
import subprocess
print(f'Pobieram {MODEL_LLM}...')
result = subprocess.run(['ollama', 'pull', MODEL_LLM], check=True)
print('Bielik gotowy.')

In [ ]:
# [2.4] Pobranie modelu EmbeddingGemma (~0.5 GB) — pierwsze pobranie poniżej minuty
import subprocess
print(f'Pobieram {MODEL_EMB}...')
result = subprocess.run(['ollama', 'pull', MODEL_EMB], check=True)
print('EmbeddingGemma gotowa.')

# Lista zainstalowanych modeli
!ollama list

In [ ]:
# [2.5] Test Bielika — pierwsze polskie zapytanie (bez RAG, bez kontekstu)
import requests, time, json

t0 = time.time()
r = requests.post(f'{OLLAMA_URL}/api/chat', json={
    'model': MODEL_LLM,
    'messages': [{'role': 'user', 'content': 'Jak często powinien być mierzony poziom chloru w basenie?'}],
    'stream': False,
})
r.raise_for_status()
data = r.json()
elapsed_ms = int((time.time() - t0) * 1000)
print(json.dumps({
    'odpowiedz': data['message']['content'],
    'model': MODEL_LLM,
    'czas_ms': elapsed_ms,
}, ensure_ascii=False, indent=2))

In [ ]:
# [2.6] Test EmbeddingGemma — wektor 768-wymiarowy dla polskiego tekstu
import requests, time, json

t0 = time.time()
r = requests.post(f'{OLLAMA_URL}/api/embed', json={
    'model': MODEL_EMB,
    'input': 'Suwerenne AI po polsku — Bielik i RAG w Google Cloud',
})
r.raise_for_status()
vec = r.json()['embeddings'][0]
elapsed_ms = int((time.time() - t0) * 1000)
print(json.dumps({
    'model': MODEL_EMB,
    'wymiary': len(vec),
    'czas_ms': elapsed_ms,
    'pierwsze_5_wartosci': [round(x, 4) for x in vec[:5]],
}, ensure_ascii=False, indent=2))

> 💡 **Zwróć uwagę:** Bielik odpowiedział na podstawie ogólnej wiedzy o pływalniach. W kroku 5 zobaczysz jak ta sama odpowiedź wygląda z kontekstem RAG — bazując na konkretnej zasadzie hotelowej (reguła 12: pomiar **co równe trzy godziny**).

> 💡 **Wymiar 768** jest stały dla `embeddinggemma` — niezależnie od długości tekstu wejściowego.

## 3. Inicjalizacja lokalnej bazy wektorowej (ChromaDB) `~1 min`

Odpowiednik kroku 4 warsztatu (BigQuery + Vector Search). W wariancie Colab używamy ChromaDB — lokalnej bazy wektorowej z indeksem HNSW. Ten sam schemat: `id`, `content`, `embedding` (768d).

In [ ]:
# [3.1] Instalacja Chroma
!pip install -q chromadb

> ℹ️ **`ERROR: pip's dependency resolver...` — to NIE jest błąd, możesz przejść dalej.**
>
> ChromaDB instaluje nowszą wersję `opentelemetry-sdk` (1.42) niż chcą widzieć preinstalowane na Colab biblioteki Google (`google-adk`, `opentelemetry-exporter-gcp-logging`). Te biblioteki **nie są używane w warsztacie** — konflikt jest czysto kosmetyczny i nie wpływa na działanie Chromy ani RAG-a.
>
> Jeśli komórka pokazuje zielony tick (✓) po lewej — instalacja się udała. Idź do 3.2.

In [ ]:
# [3.2] Utworzenie kolekcji
import chromadb

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# embedding_function=None — wektory podajemy ręcznie z Ollamy (EmbeddingGemma).
# Chroma nie generuje embeddingów automatycznie — pełna kontrola, jak w wariancie GCP.
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=None,
    metadata={'hnsw:space': 'cosine'},
)

print(f'Kolekcja "{COLLECTION_NAME}" gotowa. Rekordów: {collection.count()}')

## 4. Orchestrator FastAPI (lokalny) `~2 min`

Odpowiednik kroku 5 (Cloud Run + Orchestration API). Definiujemy te same 5 endpointów (`/`, `/ingest`, `/ask`, `/ask_direct`, `/records`) — różnice:

- **Bez JWT** — Ollama lokalnie
- **Chroma** zamiast BigQuery — `collection.query(query_embeddings=..., n_results=3)` zamiast `VECTOR_SEARCH(...COSINE)`
- **uvicorn w wątku** zamiast osobnego kontenera

In [ ]:
# [4.1] Zależności orchestratora
!pip install -q fastapi uvicorn nest_asyncio python-multipart

In [ ]:
# [4.2] Definicja aplikacji FastAPI (inline, bez JWT, na Chroma)
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse
from pydantic import BaseModel
import requests, csv, io

app = FastAPI(
    title='RAG API (Bielik & EmbeddingGemma) — wariant Colab',
    description='Lokalna wersja warsztatowa: Ollama + ChromaDB',
    version='1.0.0-colab',
)

app.mount('/static', StaticFiles(directory=STATIC_DIR), name='static')

@app.get('/', include_in_schema=False)
def root():
    return FileResponse(f'{STATIC_DIR}/index.html')

def get_embedding(text: str) -> list:
    r = requests.post(f'{OLLAMA_URL}/api/embed',
                      json={'model': MODEL_EMB, 'input': text})
    r.raise_for_status()
    return r.json()['embeddings'][0]

class AskRequest(BaseModel):
    query: str

@app.post('/ingest', tags=['Ingestion'])
async def ingest_csv(file: UploadFile = File(...)):
    content = await file.read()
    reader = csv.DictReader(io.StringIO(content.decode('utf-8')))
    ids, docs, embs = [], [], []
    for row in reader:
        doc_id = row.get('id'); text = row.get('text')
        if not doc_id or not text:
            continue
        try:
            embs.append(get_embedding(text))
            ids.append(doc_id); docs.append(text)
        except Exception as e:
            print(f'Błąd embedding dla {doc_id}: {e}')
    if ids:
        # upsert: idempotentne — kolejne /ingest nie zduplikują
        collection.upsert(ids=ids, documents=docs, embeddings=embs)
    return {'status': 'success', 'inserted_count': len(ids)}

@app.post('/ask', tags=['RAG'])
async def ask(req: AskRequest):
    qe = get_embedding(req.query)
    res = collection.query(query_embeddings=[qe], n_results=3)
    context_docs = res['documents'][0]
    context_ids = res['ids'][0]
    distances = res['distances'][0]
    # Chroma zwraca cosine distance (0 = identyczny, 2 = przeciwny);
    # przeliczamy na % podobieństwa: (1 - dist) * 100, dist ∈ [0,1] dla tekstów
    scores = [round(max(0.0, 1 - d) * 100, 1) for d in distances]
    context_text = '\\n\\n'.join(context_docs)
    prompt = (
        f'Jesteś pomocnym asystentem odpowiadającym na pytania dotyczące zasad hotelowych. '
        f'Odpowiedz na poniższe pytanie bazując TYLKO na dostarczonym kontekście.\\n\\n'
        f'KONTEKST:\\n{context_text}\\n\\n'
        f'PYTANIE:\\n{req.query}'
    )
    r = requests.post(f'{OLLAMA_URL}/api/chat', json={
        'model': MODEL_LLM,
        'messages': [{'role': 'user', 'content': prompt}],
        'stream': False,
    })
    r.raise_for_status()
    answer = r.json().get('message', {}).get('content', '')
    avg = round(sum(scores) / len(scores), 1) if scores else 0.0
    return {
        'answer': answer,
        'context_used': context_docs,
        'context_ids': context_ids,
        'context_scores': scores,
        'confidence': avg,
    }

@app.post('/ask_direct', tags=['RAG'])
async def ask_direct(req: AskRequest):
    prompt = f'Odpowiedz na poniższe pytanie w sposób jasny i zwięzły:\\n\\nPYTANIE:\\n{req.query}'
    r = requests.post(f'{OLLAMA_URL}/api/chat', json={
        'model': MODEL_LLM,
        'messages': [{'role': 'user', 'content': prompt}],
        'stream': False,
    })
    r.raise_for_status()
    return {'answer': r.json().get('message', {}).get('content', '')}

@app.get('/records', tags=['Ingestion'])
async def records(limit: int = 100):
    got = collection.get(limit=limit)
    return {
        'total': len(got['ids']),
        'records': [{'id': i, 'content': c} for i, c in zip(got['ids'], got['documents'])],
    }

print('FastAPI app zdefiniowane:', [r.path for r in app.routes if hasattr(r, 'path')])

In [ ]:
# [4.3] Uruchomienie uvicorn w tle (osobny wątek)
# Używamy uvicorn.Server (nie uvicorn.run) żeby przy ponownym uruchomieniu komórki
# 4.2/4.3 zatrzymać stary serwer i wystartować nowy z aktualnym `app`.
# Bez tego Jupyter trzyma starą instancję `app` w wątku uvicorna nawet po re-definicji.
import threading, uvicorn, nest_asyncio, time, requests as _r

nest_asyncio.apply()

# Jeśli wisi serwer z poprzedniego runu — zatrzymaj go grzecznie
if '_orch_server' in globals():
    try:
        globals()['_orch_server'].should_exit = True
        time.sleep(2)
        print('Poprzedni orchestrator zatrzymany.')
    except Exception as e:
        print(f'Ostrzeżenie przy zatrzymywaniu: {e}')

config = uvicorn.Config(app, host='0.0.0.0', port=ORCH_PORT, log_level='warning')
_orch_server = uvicorn.Server(config)
threading.Thread(target=_orch_server.run, daemon=True).start()

for _ in range(20):
    try:
        _r.get(f'http://localhost:{ORCH_PORT}/docs', timeout=1)
        # Sanity check: czy uvicorn faktycznie widzi nasze endpointy?
        paths = _r.get(f'http://localhost:{ORCH_PORT}/openapi.json').json().get('paths', {})
        assert '/ingest' in paths, f'uvicorn serwuje starszą appkę — routes: {sorted(paths.keys())}'
        print(f'Orchestrator wystartował na :{ORCH_PORT}')
        print(f'Routes: {sorted(paths.keys())}')
        break
    except AssertionError:
        raise
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError('Orchestrator nie odpowiedział w 10 s')

## 5. Zasilanie bazy i pierwsze zapytania RAG `~5 min`

Odpowiednik kroku 6 warsztatu. Wgrywamy `hotel_rules.csv` przez `/ingest`, potem 3 testowe pytania `/ask`.

In [ ]:
# [5.1] Podgląd danych wejściowych
!head -5 {CSV_PATH}
print('...')
!wc -l {CSV_PATH}

In [ ]:
# [5.2] Zasilenie bazy — wyśle CSV przez /ingest, każdy wiersz wygeneruje embedding
import requests, json

with open(CSV_PATH, 'rb') as f:
    r = requests.post(f'http://localhost:{ORCH_PORT}/ingest',
                      files={'file': ('hotel_rules.csv', f, 'text/csv')})
r.raise_for_status()
print(json.dumps(r.json(), ensure_ascii=False, indent=2))
print(f'Stan kolekcji Chroma: {collection.count()} rekordów')

In [ ]:
# [5.3] Test RAG — chlor w basenie
import requests, json
r = requests.post(f'http://localhost:{ORCH_PORT}/ask',
                  json={'query': 'Jak często powinien być mierzony poziom chloru w basenie?'})
r.raise_for_status()
print(json.dumps(r.json(), ensure_ascii=False, indent=2))

> 💡 Porównaj z odpowiedzią Bielika bez RAG z kroku 2.5 — tam model odpowiedział ogólnikowo. Tutaj RAG znalazł w Chromie regułę nr 3: **co równe trzy godziny**.

In [ ]:
# [5.4] Test RAG — śniadanie i parking
import requests, json
for q in ['O której godzinie jest podawane śniadanie?',
          'Ile kosztuje parking hotelowy?']:
    r = requests.post(f'http://localhost:{ORCH_PORT}/ask', json={'query': q})
    r.raise_for_status()
    print(f'\n--- Pytanie: {q} ---')
    print(json.dumps(r.json(), ensure_ascii=False, indent=2))

## 6. Przegląd API — Swagger /docs `~3 min`

Odpowiednik kroku 7. FastAPI automatycznie generuje interaktywną dokumentację — Colab daje nam publiczny URL przez `kernel.proxyPort`.

In [ ]:
# [6.1] Publiczny URL do /docs
from google.colab.output import eval_js
proxy_base = eval_js(f'google.colab.kernel.proxyPort({ORCH_PORT})').rstrip('/')
print('Otwórz Swagger UI w nowej karcie:')
print(f'  {proxy_base}/docs')
print()
print('Lista wszystkich endpointów:')
for r in app.routes:
    if hasattr(r, 'path') and hasattr(r, 'methods'):
        for m in (r.methods or []):
            if m in ('GET', 'POST'):
                print(f'  {m:5s} {r.path}')

## 7. Web UI w komórce notebooka `~5 min`

Odpowiednik kroku 8 — interfejs graficzny porównujący odpowiedź modelu z RAG i bez RAG. W Colab osadzamy go bezpośrednio w wyjściu komórki przez `serve_kernel_port_as_iframe`.

In [ ]:
# [7.1] Web UI w iframe — RAG vs. bez RAG, obie kolumny obok siebie
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(ORCH_PORT, height='800')

**Przykładowe pytania do wpisania:**
- *Do której godziny jest otwarty basen?*
- *Czy mogę zabrać psa do hotelu?*
- *Jak połączyć się z WiFi?*

Porównaj **lewą kolumnę** (model bez RAG) z **prawą** (model + kontekst z Chromy). Prawa kolumna pokazuje także użyte fragmenty z bazy i ich procent podobieństwa.

## 8. Checkpointy + certyfikat ukończenia `~5 min`

Mechanizm punktów zachowany z warsztatu GCP — każdy checkpoint produkuje plik `cert_artifacts/checkpoint_N.enc` zaszyfrowany openssl AES-256-CBC z kluczem opartym o `WORKSHOP_EMAIL` i `WORKSHOP_PROJECT`.

**Mapowanie punktów (75 pkt łącznie):**

| Checkpoint | Walidacja Colab | Punkty |
|---|---|---|
| 1 — Projekt | repo sklonowane + GPU widoczny | 5 |
| 2 — Konfiguracja | zmienne `WORKSHOP_*` ustawione, Ollama proces żyje | 10 |
| 3 — Modele | Bielik + EmbeddingGemma w `ollama list` | 20 |
| 4 — Vector store | kolekcja Chroma istnieje | 5 |
| 5 — Orchestrator | `GET /docs` → 200 | 10 |
| 6 — RAG | `collection.count() == 19`, `/ask` → 200 | 10 |
| 7 — Przegląd API | `/`, `/docs`, `/records` → 200 | 5 |
| 8 — Web UI | `/ask`, `/ask_direct` → 200 | 10 |

In [ ]:
# [8.1] Wspólny helper — szyfrowanie checkpointu (kompatybilne z _encrypt.sh)
import hashlib, subprocess, datetime, os, pathlib

HEADER_TEXT = 'Eskadra Bielik - Misja 2 - RAG w oparciu o model Bielik i Google Cloud'
CHECKPOINT_POINTS = {1: 5, 2: 10, 3: 20, 4: 5, 5: 10, 6: 10, 7: 5, 8: 10}
CHECKPOINT_LABELS = {
    1: 'Setup Colab + GPU',
    2: 'Konfiguracja warsztatu i Ollama',
    3: 'Modele Bielik + EmbeddingGemma',
    4: 'Vector store (ChromaDB)',
    5: 'Orchestrator FastAPI',
    6: 'Zasilanie bazy i zapytania RAG',
    7: 'Przegląd API',
    8: 'Web UI w iframe',
}
CHECKPOINT_MSGS = {
    1: 'Colab gotowy, GPU aktywny!',
    2: 'Konfiguracja i Ollama dziala. Czas na modele!',
    3: 'Oba modele zaladowane na GPU. Najtrudniejszy krok za Toba!',
    4: 'Baza wektorowa Chroma gotowa.',
    5: 'API Orchestration w tle. System RAG zlozony!',
    6: 'Wyszukiwanie semantyczne dziala. Jeden krok do mety!',
    7: 'Architektura przejrzana i zrozumiana. Ostatni krok!',
    8: 'WARSZTAT UKONCZONY! Wygeneruj certyfikat i pochwal sie wynikiem.',
}

def _key():
    """SHA512(HEADER|project|account) — identycznie jak _encrypt.sh::_checkpoint_save."""
    raw = f'{HEADER_TEXT}|{WORKSHOP_PROJECT}|{WORKSHOP_EMAIL}'
    return hashlib.sha512(raw.encode('utf-8')).hexdigest()

def save_checkpoint(num, content):
    """Tworzy cert_artifacts/checkpoint_N.enc w formacie zgodnym z wariantem GCP."""
    key = _key()
    ts = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    proc = subprocess.run(
        ['openssl', 'enc', '-aes-256-cbc', '-pbkdf2', '-iter', '100000',
         '-pass', f'pass:{key}', '-base64'],
        input=content.encode('utf-8'),
        capture_output=True,
        check=True,
    )
    encrypted = proc.stdout.decode('utf-8').strip()
    path = pathlib.Path(CERT_DIR) / f'checkpoint_{num}.enc'
    path.write_text(
        f'PROJECT_ID: {WORKSHOP_PROJECT}\n'
        f'ACCOUNT: {WORKSHOP_EMAIL}\n'
        f'CHECKPOINT: {num}\n'
        f'TIMESTAMP: {ts}\n'
        f'---BEGIN ENCRYPTED---\n{encrypted}\n---END ENCRYPTED---\n'
    )
    earned = sum(CHECKPOINT_POINTS[i] for i in range(1, 9)
                 if (pathlib.Path(CERT_DIR) / f'checkpoint_{i}.enc').exists())
    total = sum(CHECKPOINT_POINTS.values())
    width = 30; filled = earned * width // total
    bar = '[' + '#' * filled + '.' * (width - filled) + f'] {earned * 100 // total}%'
    print('=' * 54)
    print(f'  CHECKPOINT {num} ZALICZONY!')
    print(f'  {CHECKPOINT_LABELS[num]}')
    print('=' * 54)
    print(f'  Punkty za ten krok : +{CHECKPOINT_POINTS[num]} pkt')
    print(f'  Lacznie            : {earned} / {total} pkt')
    print(f'  Postep             : {bar}')
    print('=' * 54)
    print(f'  {CHECKPOINT_MSGS[num]}')
    print(f'  Artefakt           : cert_artifacts/checkpoint_{num}.enc')
    print('=' * 54)

print('Helper checkpointów gotowy.')

In [ ]:
# [8.2] Checkpoint 1 — Setup Colab + GPU
import subprocess, json
errors = []
gpu_info = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
if gpu_info.returncode == 0 and 'GPU' in gpu_info.stdout:
    print('[OK] GPU widoczny:', gpu_info.stdout.strip().split('\n')[0])
else:
    print('[!!] GPU niedostępny — włącz GPU runtime'); errors.append('gpu')

if pathlib.Path(REPO_DIR).exists():
    print(f'[OK] Repo sklonowane: {REPO_DIR}')
else:
    print('[!!] Brak repo'); errors.append('repo')

if not errors:
    save_checkpoint(1, f'CHECKPOINT_1_COLAB_SETUP\nemail={WORKSHOP_EMAIL}\nproject={WORKSHOP_PROJECT}\ngpu={gpu_info.stdout.strip()}\nverification=PASSED')
else:
    print(f'\n[!!] Błędy: {errors}. Popraw i uruchom ponownie.')

In [ ]:
# [8.3] Checkpoint 2 — Konfiguracja warsztatu i Ollama
import requests
errors = []
if WORKSHOP_EMAIL and '@' in WORKSHOP_EMAIL and WORKSHOP_EMAIL != 'uczestnik@example.com':
    print(f'[OK] WORKSHOP_EMAIL ustawione: {WORKSHOP_EMAIL}')
else:
    print('[!!] WORKSHOP_EMAIL — zmień placeholder na swój email'); errors.append('email')

try:
    requests.get(f'{OLLAMA_URL}/api/tags', timeout=2).raise_for_status()
    print(f'[OK] Ollama nasłuchuje na {OLLAMA_URL}')
except Exception as e:
    print(f'[!!] Ollama nie działa: {e}'); errors.append('ollama')

if not errors:
    save_checkpoint(2, f'CHECKPOINT_2_KONFIGURACJA\nemail={WORKSHOP_EMAIL}\nproject={WORKSHOP_PROJECT}\nollama_url={OLLAMA_URL}\nverification=PASSED')
else:
    print(f'\n[!!] Błędy: {errors}')

In [ ]:
# [8.4] Checkpoint 3 — Modele Bielik + EmbeddingGemma
import requests
errors = []
tags = requests.get(f'{OLLAMA_URL}/api/tags').json()
names = [m['name'] for m in tags.get('models', [])]
if any(MODEL_LLM in n or MODEL_LLM.split(':')[0] in n for n in names):
    print(f'[OK] Model LLM dostępny: {MODEL_LLM}')
else:
    print(f'[!!] Brak {MODEL_LLM} — uruchom sekcję 2.3'); errors.append('llm')

if any(MODEL_EMB in n for n in names):
    print(f'[OK] Model EMB dostępny: {MODEL_EMB}')
else:
    print(f'[!!] Brak {MODEL_EMB} — uruchom sekcję 2.4'); errors.append('emb')

if not errors:
    save_checkpoint(3, f'CHECKPOINT_3_MODELE\nllm={MODEL_LLM}\nemb={MODEL_EMB}\nverification=PASSED')
else:
    print(f'\n[!!] Błędy: {errors}')

In [ ]:
# [8.5] Checkpoint 4 — Vector store
errors = []
try:
    c = chroma_client.get_collection(COLLECTION_NAME)
    print(f'[OK] Kolekcja "{COLLECTION_NAME}" istnieje')
    print(f'[OK] Konfiguracja przestrzeni: {c.metadata}')
except Exception as e:
    print(f'[!!] Brak kolekcji: {e}'); errors.append('coll')

if not errors:
    save_checkpoint(4, f'CHECKPOINT_4_VECTOR_STORE\ncollection={COLLECTION_NAME}\nspace=cosine\ndimensions=768\nverification=PASSED')
else:
    print(f'\n[!!] Błędy: {errors}')

In [ ]:
# [8.6] Checkpoint 5 — Orchestrator FastAPI
import requests
errors = []
for ep in ['/docs', '/']:
    try:
        r = requests.get(f'http://localhost:{ORCH_PORT}{ep}', timeout=5)
        if r.status_code == 200:
            print(f'[OK] {ep} → HTTP 200')
        else:
            print(f'[!!] {ep} → HTTP {r.status_code}'); errors.append(ep)
    except Exception as e:
        print(f'[!!] {ep} — {e}'); errors.append(ep)

if not errors:
    save_checkpoint(5, f'CHECKPOINT_5_ORCHESTRATOR\nport={ORCH_PORT}\nendpoints_ok=docs,root\nverification=PASSED')
else:
    print(f'\n[!!] Błędy: {errors}')

In [ ]:
# [8.7] Checkpoint 6 — Dane w bazie + zapytania RAG
import requests
errors = []
count = collection.count()
if count == 19:
    print(f'[OK] Rekordy w Chroma: {count} (oczekiwano 19)')
else:
    print(f'[!!] Rekordy w Chroma: {count} (oczekiwano 19) — uruchom /ingest'); errors.append('count')

try:
    r = requests.post(f'http://localhost:{ORCH_PORT}/ask',
                      json={'query': 'Jak często mierzyć chlor?'}, timeout=60)
    if r.status_code == 200 and r.json().get('answer'):
        print(f'[OK] /ask zwrócił odpowiedź ({len(r.json()["answer"])} znaków)')
    else:
        print(f'[!!] /ask → {r.status_code}'); errors.append('ask')
except Exception as e:
    print(f'[!!] /ask błąd: {e}'); errors.append('ask')

if not errors:
    save_checkpoint(6, f'CHECKPOINT_6_RAG\nrecords={count}\ndimensions=768\nverification=PASSED')
else:
    print(f'\n[!!] Błędy: {errors}')

In [ ]:
# [8.8] Checkpoint 7 — Przegląd API
import requests
errors = []
for ep in ['/', '/docs', '/records']:
    try:
        r = requests.get(f'http://localhost:{ORCH_PORT}{ep}', timeout=5)
        if r.status_code == 200:
            print(f'[OK] GET {ep} → 200')
        else:
            print(f'[!!] GET {ep} → {r.status_code}'); errors.append(ep)
    except Exception as e:
        print(f'[!!] GET {ep} — {e}'); errors.append(ep)

if not errors:
    save_checkpoint(7, f'CHECKPOINT_7_API\nendpoints_ok=root,docs,records\nverification=PASSED')
else:
    print(f'\n[!!] Błędy: {errors}')

In [ ]:
# [8.9] Checkpoint 8 — Web UI (RAG vs. bez RAG)
import requests
errors = []
for ep in ['/ask', '/ask_direct']:
    try:
        r = requests.post(f'http://localhost:{ORCH_PORT}{ep}',
                          json={'query': 'Ile kosztuje parking?'}, timeout=60)
        if r.status_code == 200:
            print(f'[OK] POST {ep} → 200')
        else:
            print(f'[!!] POST {ep} → {r.status_code}'); errors.append(ep)
    except Exception as e:
        print(f'[!!] POST {ep} — {e}'); errors.append(ep)

if not errors:
    save_checkpoint(8, f'CHECKPOINT_8_WEB_UI\nendpoints_ok=ask,ask_direct\nverification=PASSED')
    print()
    print('=' * 54)
    print('  OSTATNI KROK — CERTYFIKAT UKONCZENIA')
    print('=' * 54)
    print('  Uruchom kolejną komórkę aby wygenerować certyfikat.')
    print('=' * 54)
else:
    print(f'\n[!!] Błędy: {errors}')

In [ ]:
# [8.10] Certyfikat ukończenia warsztatu
import pathlib, hashlib

missing = [i for i in range(1, 9)
           if not (pathlib.Path(CERT_DIR) / f'checkpoint_{i}.enc').exists()]
if missing:
    print(f'[!!] Brakuje checkpointów: {missing}. Wróć do sekcji 8.x i wykonaj je.')
else:
    print('=' * 54)
    print(' CERTYFIKAT UKOŃCZENIA — Eskadra Bielik Misja 2 (Colab)')
    print('=' * 54)
    print()
    print('Weryfikacja checkpointów:')
    for i in range(1, 9):
        p = pathlib.Path(CERT_DIR) / f'checkpoint_{i}.enc'
        size = p.stat().st_size
        h = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
        print(f'  [OK] Checkpoint {i} — {size} bajtów — sha256={h}...')
    print()
    print(' ██████╗ ██████╗  █████╗ ████████╗██╗   ██╗██╗      █████╗  ██████╗     ██╗███████╗')
    print('██╔════╝ ██╔══██╗██╔══██╗╚══██╔══╝██║   ██║██║     ██╔══██╗██╔════╝     ██║██╔════╝')
    print('██║  ███╗██████╔╝███████║   ██║   ██║   ██║██║     ███████║██║          ██║█████╗')
    print('██║   ██║██╔══██╗██╔══██║   ██║   ██║   ██║██║     ██╔══██║██║     ██   ██║██╔══╝')
    print('╚██████╔╝██║  ██║██║  ██║   ██║   ╚██████╔╝███████╗██║  ██║╚██████╗╚█████╔╝███████╗')
    print(' ╚═════╝ ╚═╝  ╚═╝╚═╝  ╚═╝   ╚═╝    ╚═════╝ ╚══════╝╚═╝  ╚═╝ ╚═════╝ ╚════╝ ╚══════╝')
    print()
    print('         🚀  WARSZTAT UKOŃCZONY — 75 / 75 pkt  🚀')
    print()
    print('=' * 54)
    print(f' Email uczestnika : {WORKSHOP_EMAIL}')
    print(f' Projekt          : {WORKSHOP_PROJECT}')
    print(f' Artefakty        : {CERT_DIR}')
    print('=' * 54)

## 9. Sprzątanie `~1 min`

Odpowiednik kroku 10. Na Colab nie ma kosztów do zatrzymania, ale dobrze posprzątać procesy zanim zamkniesz sesję — szczególnie jeśli planujesz uruchomić inny notebook na tym samym kernelu.

In [ ]:
# [9.1] Zatrzymanie procesów i zwolnienie GPU
import subprocess, gc

try:
    subprocess.run(['pkill', '-f', 'ollama'], check=False)
    print('[OK] Ollama zatrzymana')
except Exception as e:
    print(f'[--] {e}')

# uvicorn działa w wątku — zamknie się gdy kernel zamknie się sam.
# Chroma persistent — zostaje w /content/chroma_db.

gc.collect()
print('[OK] Sprzątanie zakończone.')
print()
print('Co warto zachować przed zamknięciem sesji:')
print(f'  cert_artifacts/  — pobierz przez panel "Files" w Colab')
print(f'  chroma_db/       — opcjonalnie, jeśli chcesz wrócić do bazy')

## Zostańmy w kontakcie

Materiał warsztatowy: [Legard777/eskadra-bielik-misja2](https://github.com/Legard777/eskadra-bielik-misja2). Wariant Colab to dodatek do oryginalnego warsztatu GCP — kod orchestratora, modele i dane są identyczne, różni się tylko warstwa runtime.